# Investigation 3 — Well Activity Radius Signal (Q3)

**Hypothesis:** Available blocks within 10–25 km of a recently spudded well receive bids at a higher rate than blocks in quiet areas.

**Method:** For each available block, count wells spudded within 10 km and 25 km in the prior 6 and 18 months. Bin by well count, compute bid rates.

**Success:** Clear monotonic gradient (more wells → higher bid rate), p < 0.05.

---
*Prototype using Sale 247 (March 2017). Swap to Dec 2025 for final validation.*

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

ROOT = os.path.abspath(os.path.join("..", ".."))
sys.path.insert(0, os.path.join(ROOT, "mvp0"))
import boem_loader as bl

SALE_DATE = pd.Timestamp("2017-03-22")
RADII_KM = [10, 25]
WINDOWS_MO = [6, 18]
print(f"Sale date: {SALE_DATE.date()}")
print(f"Radii: {RADII_KM} km")
print(f"Lookback windows: {WINDOWS_MO} months")

## 1. Load data

In [ ]:
sales  = bl.load_master_sales(os.path.join(ROOT, "data/lease-sales/master_lease_sales.csv"))
blocks = bl.load_blocks(os.path.join(ROOT, "data/shapefiles/blocks.shp"), to_utm=True)
wells  = bl.load_boreholes(os.path.join(ROOT, "data/wells/mv_boreholes_all.txt"), region="G", to_utm=True)

print(f"Sale bids: {len(sales)}")
print(f"Blocks: {len(blocks)}")
print(f"Wells (GOM): {len(wells)}")

## 2. Build sale universe and block centroids

In [ ]:
sale_prots = set(sales["Protraction_ID"].unique())
universe = blocks[blocks["Protraction_ID"].isin(sale_prots)].copy().reset_index(drop=True)
print(f"Universe: {len(universe)} blocks")

# Mark which blocks received bids
bid_pairs = set(zip(sales["Protraction_ID"], sales["Block_Number"]))
universe["did_bid"] = [
    (r["Protraction_ID"], r["Block_Number"]) in bid_pairs
    for _, r in universe.iterrows()
]
print(f"Blocks with bids: {universe['did_bid'].sum()}")

# Block centroids (UTM meters) for distance calculations
universe["centroid"] = universe.geometry.centroid

## 3. Filter wells by lookback window

In [ ]:
well_sets = {}
for mo in WINDOWS_MO:
    start = SALE_DATE - pd.DateOffset(months=mo)
    mask = (wells["Spud_Date"] >= start) & (wells["Spud_Date"] <= SALE_DATE)
    well_sets[mo] = wells[mask].copy()
    print(f"Wells spudded in prior {mo} months: {len(well_sets[mo])}")

## 4. Count wells within each radius of each block

For each (radius, window) combination, buffer each well point and spatial-join to block centroids.

In [ ]:
%%time
# Create centroid GeoDataFrame for spatial joins
centroids = gpd.GeoDataFrame(
    universe[["Protraction_ID", "Block_Number", "did_bid"]],
    geometry=universe["centroid"],
    crs=universe.crs,
)

results = {}
for mo in WINDOWS_MO:
    ws = well_sets[mo]
    for radius_km in RADII_KM:
        radius_m = radius_km * 1000
        label = f"{radius_km}km_{mo}mo"

        # Buffer wells
        buffered = ws.copy()
        buffered["geometry"] = buffered.geometry.buffer(radius_m)

        # Spatial join: which block centroids fall within each well buffer?
        joined = gpd.sjoin(centroids, buffered[["geometry"]], how="left", predicate="within")

        # Count wells per block
        well_count = (
            joined.groupby(joined.index)["index_right"]
            .count()
            .reindex(range(len(universe)), fill_value=0)
        )
        universe[f"wells_{label}"] = well_count.values
        results[label] = universe[f"wells_{label}"]

        n_with = (well_count > 0).sum()
        print(f"{label}: {n_with} blocks have >= 1 well nearby")

## 5. Bin blocks and compute bid rates per bin

In [ ]:
combo_results = []

for label in results:
    col = f"wells_{label}"
    universe["well_bin"] = pd.cut(
        universe[col], bins=[-1, 0, 1, 999], labels=["0 wells", "1 well", "2+ wells"]
    )

    rates = universe.groupby("well_bin", observed=True)["did_bid"].agg(["sum", "count", "mean"])
    rates.columns = ["bids", "blocks", "bid_rate"]

    # Spearman correlation between well count and bid
    mask = universe[col].notna()
    rho, p = spearmanr(universe.loc[mask, col], universe.loc[mask, "did_bid"])

    combo_results.append({
        "combination": label,
        "rate_0": rates.loc["0 wells", "bid_rate"] if "0 wells" in rates.index else 0,
        "rate_1": rates.loc["1 well", "bid_rate"] if "1 well" in rates.index else 0,
        "rate_2plus": rates.loc["2+ wells", "bid_rate"] if "2+ wells" in rates.index else 0,
        "spearman_rho": rho,
        "p_value": p,
    })

    print(f"\n=== {label} ===")
    display(rates)
    print(f"Spearman rho = {rho:.4f}, p = {p:.2e}")

## 6. Summary heatmap: (radius × lookback window)

In [ ]:
summary = pd.DataFrame(combo_results)
display(summary)

# Find strongest combination
best = summary.loc[summary["spearman_rho"].abs().idxmax()]
print(f"\nStrongest combination: {best['combination']}")
print(f"  Spearman rho = {best['spearman_rho']:.4f}, p = {best['p_value']:.2e}")

In [ ]:
# Heatmap of bid rates by (radius, window, well_bin)
fig, axes = plt.subplots(1, len(RADII_KM), figsize=(14, 5), sharey=True)

for i, radius_km in enumerate(RADII_KM):
    ax = axes[i]
    for mo in WINDOWS_MO:
        label = f"{radius_km}km_{mo}mo"
        col = f"wells_{label}"
        universe["well_bin"] = pd.cut(
            universe[col], bins=[-1, 0, 1, 999], labels=["0", "1", "2+"]
        )
        rates = universe.groupby("well_bin", observed=True)["did_bid"].mean() * 100
        ax.plot(rates.index, rates.values, "o-", label=f"{mo}-month window", markersize=8)

    ax.set_title(f"Radius: {radius_km} km")
    ax.set_xlabel("Wells in radius")
    ax.set_ylabel("Bid rate (%)" if i == 0 else "")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Well Activity Signal: Bid Rate by Nearby Well Count", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Lag analysis: 0–6 months vs. 7–18 months

In [ ]:
# Use the best radius, split the 18-month window into recent vs. older
best_radius = int(best["combination"].split("km")[0])
radius_m = best_radius * 1000

cutoff_6mo = SALE_DATE - pd.DateOffset(months=6)
cutoff_18mo = SALE_DATE - pd.DateOffset(months=18)

recent_wells = wells[(wells["Spud_Date"] >= cutoff_6mo) & (wells["Spud_Date"] <= SALE_DATE)]
older_wells  = wells[(wells["Spud_Date"] >= cutoff_18mo) & (wells["Spud_Date"] < cutoff_6mo)]

for lag_label, ws in [("0–6 months", recent_wells), ("7–18 months", older_wells)]:
    buffered = ws.copy()
    buffered["geometry"] = buffered.geometry.buffer(radius_m)
    joined = gpd.sjoin(centroids, buffered[["geometry"]], how="left", predicate="within")
    wc = joined.groupby(joined.index)["index_right"].count().reindex(range(len(universe)), fill_value=0)
    has_well = wc > 0
    rate_with = universe.loc[has_well.values, "did_bid"].mean()
    rate_without = universe.loc[~has_well.values, "did_bid"].mean()
    print(f"{lag_label} ({best_radius}km): with_well={rate_with:.4%} vs without={rate_without:.4%}  "
          f"(lift={rate_with/rate_without:.1f}x)" if rate_without > 0 else "")

## 8. Map visualization

In [ ]:
best_col = f"wells_{best['combination']}"

fig, ax = plt.subplots(figsize=(12, 8))
universe.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.2)

# Shade blocks by well count
has_wells = universe[universe[best_col] > 0]
has_wells.plot(ax=ax, column=best_col, cmap="YlOrRd", edgecolor="gray",
               linewidth=0.3, legend=True, legend_kwds={"label": "Wells in radius"})

# Overlay bid blocks
bid_blocks = universe[universe["did_bid"]]
bid_blocks.plot(ax=ax, facecolor="none", edgecolor="blue", linewidth=1.5, label="Bid placed")

# Plot well locations
best_mo = int(best["combination"].split("_")[1].replace("mo", ""))
ws = well_sets[best_mo]
ws.plot(ax=ax, color="black", markersize=3, alpha=0.5, label="Wells")

ax.set_title(f"Well Activity Signal — {best['combination']} (Sale 247)")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## 9. Interpretation

| Result | Interpretation | Action |
|--------|---------------|--------|
| Clear monotonic gradient, p < 0.05 | Signal real. Lock in best radius/window for MVP 1. | Proceed with Treasure Toggle. |
| Gradient exists but weak | Signal marginal. Directional only. | Keep Treasure Toggle, lower accuracy target to 60%. |
| No gradient | Well activity doesn't predict bidding at block level. | Remove Treasure Toggle from MVP 2. |

**Note:** Prototype uses Sale 247. Final validation should use Dec 2025.